In [1]:
#MOana specific
import os, subprocess
print("PID:", os.getpid())
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("hostname:", subprocess.run(["hostname"], capture_output=True, text=True).stdout.strip())
print(subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout)


PID: 9323
CUDA_VISIBLE_DEVICES: None
hostname: 3c1b30cc1434
GPU 0: NVIDIA TITAN X (Pascal) (UUID: GPU-3f2d65bc-b02a-fda5-51fc-72ed6e13beed)
GPU 1: NVIDIA TITAN X (Pascal) (UUID: GPU-d3a09050-15e1-11e6-e9ca-02fad034f13a)
GPU 2: NVIDIA TITAN X (Pascal) (UUID: GPU-445803e7-81b2-f9b2-fb68-27d1e934937a)
GPU 3: NVIDIA TITAN X (Pascal) (UUID: GPU-6044dc59-ec99-d562-5689-4099eb7024b0)



In [2]:

#MOANA ONLY
#os.environ["CUDA_VISIBLE_DEVICES"] = "1"
#check for errors
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"


In [1]:
# Cell 0 — CONFIG
from pathlib import Path
import numpy as np
%load_ext autoreload
%autoreload 2
import sys, os
from dotenv import load_dotenv
if sys.platform == "win32":
    project_root = Path(r"F:/MScPROJECT_LOCAL")
else:
    _linux_candidates = [Path("/knuckles_drive"), Path("/workspaces/MScPROJECT_LOCAL")]
    for _candidate in _linux_candidates:
        if _candidate.exists():
            project_root = _candidate
            break
    else:
        from A_Config import REPO_ROOT
        project_root = Path(REPO_ROOT)  # robust fallback -- REPO_ROOT is computed from A_Config.py's own location, so it's correct on any machine/mount
load_dotenv(project_root / ".env")
case_name = "201_copper_01"
#experiment = "cat_03_agent_temp"
experiment = None
Question = "what weapoon was the suspect holding?"
if experiment:
    asset_name = f"{case_name}_{experiment}"
else: 
    asset_name = case_name
from A_Config import (
    set_case, case_dir, source_dir, query_dir, yolo_masks_dir,
    frames_for_cam_poses_dir, frames_for_recon_dir, vggt_o_output_dir,
    predictions_path, cut3r_output_dir, ply_dir, reg_dir,
    csv_path_gps, csv_path_pose, FRAME_NAME_FMT,
)
set_case(case_name, experiment)

#TEMP
#This needs to be filed after user input decides on what to analyse
ppt_template = '03_Version_1'

# video loading
# Auto-find the video in 010_source
videos = list(source_dir().glob("*.mp4"))
assert len(videos) == 1, f"Expected 1 video, found {len(videos)}: {videos}"
video_path = videos[0]
print(f"Case    : {case_name}")
print(f"Video   : {video_path.name}")

#reporting
from C_CSV_report import add_to_report
overview = {
    "Video_name"      : video_path.name, 
    "Title"      : f"{case_name}_{experiment}"
}
add_to_report(overview)
api_key = os.getenv("STREETVIEW_API_KEY")

Case    : 201_copper_01
Video   : log_1081378_body-cam_v1.mp4


In [2]:
#FLAGS dict and 'producer dict
flags = {
    "VID_TO_TEXT": True,
    "TRACKING": True,
    "FRAME_SELECTION": False,
    "RECON_CAM_POSES": False,
    "RECON_3D": False,
    "CUT3R_RECON": False,
    "MEGA_SAM_RECON": False,
    "VGGT_RECON": False,
    "VGGT_O_RECON": False,
    "MULTI_PART_RECON": False,
    "GOOGLE_MAP": False,
    "PROJECTION_MAP": False,
    "MAKE_MOVIE": False,
    "MAKE_MOVIE_CUTS" : True,
    "MASK_OVERLAYS": True ,
    "PHYSICS_OVERLAYS" : False,
    "MULTICAM" : False
}
MENU_KEYS = {"RECON_CAM_POSES", "RECON_3D", "CUT3R_RECON", "MEGA_SAM_RECON",
             "VGGT_RECON", "VGGT_O_RECON", "MULTI_PART_RECON",
             "GOOGLE_MAP", "PROJECTION_MAP", "MASK_OVERLAYS", "MULTICAM"}
producer_flags = {k: v for k, v in flags.items() if k in MENU_KEYS}

In [ ]:
#GEMINI video ->  text 
if flags["VID_TO_TEXT"] == True:
    from Two2D.A_gemini import gemini_vid_to_text
    gem_summaries = gemini_vid_to_text(video_path, case_name, query_dir())
else:
    print("Cell disabled")



In [3]:
#2d ANALYSIS - JUST GPS now!
from D_2d_analysis import analysis_2D
analysis_2d_for_decisions = analysis_2D(video_path,  api_key)

No location metadata in /workspaces/knuckles_drive/Data/201_copper_01/010_source/log_1081378_body-cam_v1.mp4


In [3]:
#PRODUCER DRAFT 1
sys.path.insert(0, str(project_root / "src" / "claude_agent"))
from producer_agent_execution import run_producer_agent, build_producer_prompt_draft1
analysis_2d_for_decisions=[]
prompt = build_producer_prompt_draft1(Question, analysis_2d_for_decisions, producer_flags)
paper_edit_json_path = run_producer_agent(prompt, mode ="draft")


/workspaces/knuckles_drive/Data/201_copper_01/012_agent_p_output
[13:13:42 +  0.00s] run_producer_agent: start
[13:13:42 +  0.00s] thread: start
[13:13:42 +  0.00s] event loop: created
[13:13:42 +  0.01s] query: start


Ignoring 5 permissions.allow entries from .claude/settings.json: this workspace has not been trusted. Run Claude Code interactively here once and accept the trust dialog, or set projects["/workspaces/knuckles_drive"].hasTrustDialogAccepted: true in /home/vscode/.claude.json.
Ignoring 1 permissions.additionalDirectories entry from .claude/settings.json: this workspace has not been trusted. Run Claude Code interactively here once and accept the trust dialog, or set projects["/workspaces/knuckles_drive"].hasTrustDialogAccepted: true in /home/vscode/.claude.json.


[13:14:07 + 25.37s] turn 1 usage (claude-sonnet-5): in=2 out=2 cache_write=13573 cache_read=3372
[13:14:22 + 39.80s] turn 2 usage (claude-sonnet-5): in=2 out=2 cache_write=13573 cache_read=3372
[13:14:22 + 39.80s] tool call: Write (/workspaces/knuckles_drive/Data/201_copper_01/012_agent_p_output/201_copper_01_paper_edit_draft.json)
[13:14:27 + 44.79s] turn 3 usage (claude-sonnet-5): in=2 out=2 cache_write=4034 cache_read=16945
[13:14:27 + 44.83s] result: status=success num_turns=2 duration_ms=43993 duration_api_ms=45246 cost_usd=0.1810891
[13:14:27 + 44.83s] result usage (aggregate): {'input_tokens': 4, 'cache_creation_input_tokens': 17607, 'cache_read_input_tokens': 20317, 'output_tokens': 4275, 'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0}, 'service_tier': 'standard', 'cache_creation': {'ephemeral_1h_input_tokens': 17607, 'ephemeral_5m_input_tokens': 0}, 'inference_geo': 'not_available', 'iterations': [{'input_tokens': 2, 'output_tokens': 319, 'cache_read_inp

In [5]:
# FORMAT PRODUCER OUTPUT into HTML and FLAGS
import importlib
import json
sys.path.insert(0, str(project_root / "src" / "claude_agent"))
import render_paper_edit
importlib.reload(render_paper_edit)

from render_paper_edit import render_and_save, derive_and_save_flags

html_path = render_and_save(paper_edit_json_path)
print(f"\n=== Rendered HTML ===\n  {html_path}")

flags_path = derive_and_save_flags(paper_edit_json_path, producer_flags.keys())
producer_output_flags = json.loads(flags_path.read_text(encoding="utf-8"))
print(f"\n=== Derived flags (from {flags_path.name}) ===")
print(json.dumps(producer_output_flags, indent=2))


=== Rendered HTML ===
  /workspaces/knuckles_drive/Data/201_copper_01/012_agent_p_output/201_copper_01_draft_paper_edit.html

=== Derived flags (from 201_copper_01_draft_flags.json) ===
{
  "RECON_CAM_POSES": false,
  "RECON_3D": false,
  "CUT3R_RECON": false,
  "MEGA_SAM_RECON": false,
  "VGGT_RECON": false,
  "VGGT_O_RECON": false,
  "MULTI_PART_RECON": false,
  "GOOGLE_MAP": false,
  "PROJECTION_MAP": false,
  "MASK_OVERLAYS": true,
  "MULTICAM": false
}


In [ ]:
#Get key things to examine (DEFUNCT)
if flags["VID_TO_TEXT"] ==True:
    #Query to text to get rough frame ranges. 
    from Two2D.A_gemini import query_from_description
    human_out, detections = query_from_description(
        Question ,
        query_dir() / f"{case_name}_description.txt" , 
        query_dir() / f"{case_name}_objects.txt",
        case_dir()
        
    )
    print(detections)
else:
    print("Cell disabled")

In [7]:
#ROBOFLOW TRACKING SAM3
if flags["MASK_OVERLAYS"]:
    sys.path.insert(0, str(project_root / "src" / "claude_agent"))
    from A_ROBOFLOW_SAM3 import track_subject_sam3
    analysis_2d_for_decisions.update (track_subject_sam3(video_path)) # add to the dict
    tracker = "SAM3"
    print(analysis_2d_for_decisions)
else:
    print("Cell disabled")


accessing data
initialising tracking
[beat 2] making subclip weapon
[beat 2] requested 147-155s -> keyframe-aligned start frame 4320


[beat 2] no detections across any requested span
[beat 4] making subclip weapon
[beat 4] requested 225-237s -> keyframe-aligned start frame 6660
{2: {'Subject_frames_present': [], 'subject_first_frame': None, 'subject_last_frame': None, 'continuous frame sequences': [], 'best_seq_idx': None, 'masks_dir': '/workspaces/knuckles_drive/Data/201_copper_01/015_SAM3_masks/beat02'}, 4: {'Subject_frames_present': [6712, 6713, 6714, 6715, 6716, 6717, 6718, 6719, 6720, 6728, 6729, 6730, 6731, 6732, 6733, 6734, 6735, 6736, 6737, 6738, 6739, 6740, 6741, 6742, 6743, 6744, 6745, 6746, 6747, 6748, 6749, 6750, 6751, 6752, 6753, 6754, 6755, 6756, 6757, 6758, 6759, 6760, 6761, 6762, 6763, 6764, 6765, 6766, 6767, 6768, 6769, 6770, 6771, 6772, 6802, 6803, 7002, 7003, 7006, 7007, 7008, 7009, 7010, 7011, 7012, 7013, 7014, 7015, 7016, 7017, 7018, 7019, 7020, 7021, 7022, 7023, 7024, 7025, 7026, 7027, 7028, 7029, 7030, 7031, 7032, 7033, 7034, 7035, 7036, 7037, 7038, 7039, 7040, 7041, 7042, 7043, 7044, 7045, 704

In [6]:
#run this if restarting kernel
from A_Config import sam3_masks_dir
from D_2d_analysis import mask_analysis
from D_2d_analysis import analysis_2d_from_masks
paper_edit_json_path = case_dir() / "012_agent_p_output" / f"{case_name}_paper_edit_draft.json"
analysis_2d_for_decisions = analysis_2d_from_masks(paper_edit_json_path)
print(analysis_2d_for_decisions)


no masks in /workspaces/knuckles_drive/Data/201_copper_01/015_SAM3_masks/beat02
no masks in /workspaces/knuckles_drive/Data/201_copper_01/015_SAM3_masks/beat02
no masks in /workspaces/knuckles_drive/Data/201_copper_01/015_SAM3_masks/beat04
no masks in /workspaces/knuckles_drive/Data/201_copper_01/015_SAM3_masks/beat04
{2: {'Subject_frames_present': [], 'subject_first_frame': None, 'subject_last_frame': None, 'subject_duration_frames': None, 'continuous frame sequences': [], 'best_seq_idx': None, 'masks_dir': '/workspaces/knuckles_drive/Data/201_copper_01/015_SAM3_masks/beat02'}, 4: {'Subject_frames_present': [], 'subject_first_frame': None, 'subject_last_frame': None, 'subject_duration_frames': None, 'continuous frame sequences': [], 'best_seq_idx': None, 'masks_dir': '/workspaces/knuckles_drive/Data/201_copper_01/015_SAM3_masks/beat04'}}


In [ ]:
#subject sselector NOT IN USE
if flags["VID_TO_TEXT"] ==True:
        
    from A_Interactions import subject_selector
    subject_frame_range = subject_selector(
        video_path,
        detections,
        conf= 0.05,
        ) 
    print(subject_frame_range)
else:
    print("Cell disabled")

In [ ]:
#YOLO - track moving subject footage NOT IN USE
#works from video
from A_YOLO_seg import track_subject_masks_from_hints
if flags["TRACKING"] == True:
    import sys
    sys.path.insert(0, str(project_root / "src"))

    from A_YOLO_seg import track_subject_masks

    results = track_subject_masks_from_hints(
        video_path=video_path,
        output_dir=str(yolo_masks_dir()),
        detections=subject_frame_range,
        miss_threshold=100,
        conf=0.05
        )

    print(f"\n{len(results)} masks written to {yolo_masks_dir()}")
    print(results)
else:
    print("Cell disabled")


No location metadata in /workspaces/knuckles_drive/Data/201_copper_01/010_source/log_1081378_body-cam_v1.mp4


In [ ]:
#START AND END CALCULATORS
print(min(analysis['Subject_frames_present']))
print(max(analysis['Subject_frames_present']))

best_idx = analysis['best_seq_idx']
start_3D_recon = analysis['continuous frame sequences'][best_idx][0]
end_3D_recon = analysis['continuous frame sequences'][-1][1]      

print("start 3d", start_3D_recon)
print("end 3d" ,end_3D_recon)

start_cam_recon = max(start_3D_recon-2200, 0) # this needs to be made batter!
end_cam_recon   = min(end_3D_recon + 4500 , 9000)#this is shit

movie_start = start_3D_recon - (5* 30) #this needs to be made auto
movie_end = end_3D_recon + (5*30)

#find the limits of vggt
capacity = 100
recon_dur = end_3D_recon - start_3D_recon
recon_interframe_interval =   (recon_dur+capacity-1) // capacity
print("interval" , recon_interframe_interval)

In [8]:
#find the cut points
if flags["MAKE_MOVIE_CUTS"] == True:
    import Two2D.W_video_editor
    W_video_editor.find_all_cut_points(video_path, analysis_2d_for_decisions)

In [12]:
#run producer again
sys.path.insert(0, str(project_root / "src" / "claude_agent"))
from producer_agent_execution import run_producer_agent, build_producer_prompt_revision

prompt = build_producer_prompt_revision(Question, analysis_2d_for_decisions, producer_flags)
paper_edit_json_path = run_producer_agent(prompt)


[13:51:18 +  0.00s] run_producer_agent: start
[13:51:18 +  0.00s] thread: start
[13:51:18 +  0.00s] event loop: created
[13:51:18 +  0.01s] query: start


Ignoring 5 permissions.allow entries from .claude/settings.json: this workspace has not been trusted. Run Claude Code interactively here once and accept the trust dialog, or set projects["/workspaces/knuckles_drive"].hasTrustDialogAccepted: true in /home/vscode/.claude.json.
Ignoring 1 permissions.additionalDirectories entry from .claude/settings.json: this workspace has not been trusted. Run Claude Code interactively here once and accept the trust dialog, or set projects["/workspaces/knuckles_drive"].hasTrustDialogAccepted: true in /home/vscode/.claude.json.


[13:52:06 + 48.71s] turn 1 usage (claude-sonnet-5): in=2 out=5 cache_write=16986 cache_read=3372
[13:52:21 + 63.49s] turn 2 usage (claude-sonnet-5): in=2 out=5 cache_write=16986 cache_read=3372
[13:52:21 + 63.49s] tool call: Write (/workspaces/knuckles_drive/Data/201_copper_01/012_agent_p_output/201_copper_01_paper_edit.json)
[13:52:27 + 68.99s] turn 3 usage (claude-sonnet-5): in=2 out=2 cache_write=6437 cache_read=20358
[13:52:27 + 69.03s] result: status=success num_turns=2 duration_ms=68131 duration_api_ms=69346 cost_usd=0.25722900000000004
[13:52:27 + 69.03s] result usage (aggregate): {'input_tokens': 4, 'cache_creation_input_tokens': 23423, 'cache_read_input_tokens': 23730, 'output_tokens': 6774, 'server_tool_use': {'web_search_requests': 0, 'web_fetch_requests': 0}, 'service_tier': 'standard', 'cache_creation': {'ephemeral_1h_input_tokens': 23423, 'ephemeral_5m_input_tokens': 0}, 'inference_geo': 'not_available', 'iterations': [{'input_tokens': 2, 'output_tokens': 413, 'cache_read

In [ ]:
#make mask overlays and return image sequence
from Two2D.W_video_editor import mask_compositor
tracker = "SAM3"
movie_start = 225*30 
movie_end = 237*30
mask_compositor(video_path, 
                            start = movie_start, 
                            end = movie_end, 
                            tracker= tracker, 
                            mask_start = movie_start,
                            mask_end = movie_end,
                            handles=2, 
                          color=(0, 200, 0), 
                          alpha=0.4) 


In [16]:
#Make the edited multimedia movie
if flags["MAKE_MOVIE_CUTS"] == True:
    import Two2D.W_video_editor
 
    W_video_editor.assemble_paper_edit(video_path)
else:
    print("Cell disabled")

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 14.3.0 (conda-forge gcc 14.3.0-5)
  configuration: --prefix=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_h_env_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_plac --cc=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_build_env/bin/x86_64-conda-linux-gnu-cc --cxx=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_build_env/bin/x86_64-conda-linux-gnu-c++ --nm=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_build_env/bin/x86_64-conda-linux-gnu-nm --ar=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_build_env/bin/x86_64-conda-linux-gnu-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libop

making overlay for beat {'archetype': 'EVENT_ACTION', 'order': 2, 'segment_type': 'real', 'duration_seconds': 41.7, 'source': "Bodycam, door breach and confrontation, officer shouts 'Drop the gun!' as suspect holds weapon (02:21-02:41); taser and shot deployed, suspect falls and is restrained (02:41-03:00)", 'cut_mode': 'fixed_frames', 'start_frame': 4189, 'end_frame': 5440, 'lead_in_seconds': None, 'search_window_start_seconds': None, 'search_window_end_seconds': None, 'asset_request': None, 'requested_flags': ['MASK_OVERLAYS'], 'quote': "02:30 [Officer 2]: Don't, drop the gun. / 02:33 [Officer 2]: Drop the gun! / 02:39 [Officer 1]: Hey, shots fired, taser deployed, taser deployed!", 'rationale': "R1.9 merged former EVENT_TRIGGER and EVENT_ACTION beats: both from same bodycam camera with overlapping auto-resolved frame ranges (4189-4871 and 4780-5440); merged to earlier start/later end per merge rule, archetype set to EVENT_ACTION (priority over EVENT_TRIGGER); R4.1 sync dialogue disc

ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 14.3.0 (conda-forge gcc 14.3.0-5)
  configuration: --prefix=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_h_env_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_plac --cc=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_build_env/bin/x86_64-conda-linux-gnu-cc --cxx=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_build_env/bin/x86_64-conda-linux-gnu-c++ --nm=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_build_env/bin/x86_64-conda-linux-gnu-nm --ar=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_build_env/bin/x86_64-conda-linux-gnu-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libop

making overlay for beat {'archetype': 'AFTERMATH', 'order': 3, 'segment_type': 'real', 'duration_seconds': 16.1, 'source': 'Bodycam, officer recovers weapon from floor and identifies it, 03:45-04:00', 'cut_mode': 'fixed_frames', 'start_frame': 6667, 'end_frame': 7149, 'lead_in_seconds': None, 'search_window_start_seconds': None, 'search_window_end_seconds': None, 'asset_request': None, 'requested_flags': ['MASK_OVERLAYS'], 'quote': "03:57 [Officer 1]: It's a fucking BB gun.", 'rationale': "R4.1 sync dialogue directly answers the brief; R10.1 mask overlay confirms the object present across frames 6712-7104 of this beat's span (392 of 482 frames, best continuous run 7002-7104), consistent with the audio identification as a BB gun", 'rejected_alternative': None, 'flag': None, 'tracked_subject': {'subject': 'weapon', 'spans': [{'start_s': 225, 'end_s': 237}]}}


ffmpeg version 7.1.1 Copyright (c) 2000-2025 the FFmpeg developers
  built with gcc 14.3.0 (conda-forge gcc 14.3.0-5)
  configuration: --prefix=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_h_env_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_placehold_plac --cc=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_build_env/bin/x86_64-conda-linux-gnu-cc --cxx=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_build_env/bin/x86_64-conda-linux-gnu-c++ --nm=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_build_env/bin/x86_64-conda-linux-gnu-nm --ar=/home/conda/feedstock_root/build_artifacts/ffmpeg_1758923917305/_build_env/bin/x86_64-conda-linux-gnu-ar --disable-doc --enable-openssl --enable-demuxer=dash --enable-hardcoded-tables --enable-libfreetype --enable-libharfbuzz --enable-libfontconfig --enable-libop

In [ ]:
from V_text_summary import gemini_final_report, gemini_cache_video
from A_Config import assets_dir
video_path = assets_dir() / f"{asset_name}_overlay_video.mp4"
cache =  gemini_cache_video(video_path, assets_dir(), ttl="6000s")
gemini_final_report()

In [ ]:
# Cell — Populate PowerPoint from CSV
import sys
import importlib
sys.path.insert(0, str(project_root / "src"))

import W_populate_pptx
importlib.reload(W_populate_pptx)
from W_populate_pptx import populate



report_template = project_root / "src" / "Powerpoint" / f"{ppt_template}.pptx"
ppt_assets   = case_dir() / "090_Assets"
ppt_csv      = ppt_assets / f"{case_name}.csv"
ppt_output   = case_dir() / "100_Powerpoint" / f"{case_name}.pptx"


populate(str(report_template))
